# 10 — Unified GeoPackage Schema Hardening and Metadata Integration

## Objective

Notebook 09 established the first unified GeoPackage structure for the Canadian geological CO₂ storage database. The purpose of this notebook is to **harden that schema, formalize relationships between tables, and integrate metadata and provenance directly into the GeoPackage**.

The goal is not to change the underlying geological harmonization logic unless validation identifies a structural inconsistency. Instead, this notebook focuses on making the unified GeoPackage a more complete, self-describing, and reusable database that can be inspected consistently through tools such as **DBeaver, QGIS, GeoPandas, GDAL, and SQLite**.

The main objectives are to:

1. **Validate the current canonical schema**
   - inspect the four canonical tables created in Notebook 09;
   - verify identifier uniqueness, nullability, and expected table cardinality;
   - identify orphaned or contradictory relationships between records.

2. **Formalize relational structure**
   - distinguish technical GeoPackage row identifiers from canonical domain identifiers;
   - define primary key, unique, and foreign key relationships where appropriate;
   - formalize relationships between:
     - `storage_units`,
     - `storage_features`,
     - `storage_assessments`,
     - `administrative_features`;
   - preserve valid cases where relationships are intentionally nullable, such as unit-level assessments that are not tied to a specific spatial feature.

3. **Improve GeoPackage registration**
   - confirm that spatial layers are correctly registered in the GeoPackage system tables;
   - register non-spatial canonical tables as GeoPackage attribute tables where appropriate;
   - preserve GeoPackage geometry, CRS, spatial-index, and extension infrastructure.

4. **Integrate source metadata and provenance**
   - inspect metadata available from the original source GeoPackages;
   - preserve useful information such as:
     - source dataset,
     - source layer,
     - provider,
     - citation,
     - source URL,
     - license,
     - release/version information,
     - source CRS,
     - source field definitions,
     - units,
     - methodological notes;
   - avoid copying source GeoPackage metadata records blindly when they refer to tables or layers that no longer exist in the harmonized database.

5. **Create a canonical metadata model**
   - define database-level metadata tables that can be queried directly through SQLite or DBeaver;
   - document canonical tables, fields, units, descriptions, and source lineage;
   - preserve the source-to-canonical harmonization decisions established in Notebook 09.

6. **Populate standard GeoPackage metadata structures**
   - evaluate and populate relevant standard tables such as:
     - `gpkg_contents`,
     - `gpkg_metadata`,
     - `gpkg_metadata_reference`,
     - `gpkg_data_columns`,
     - `gpkg_extensions`;
   - ensure metadata can be accessed through GIS software where supported.

7. **Validate the hardened database**
   - check foreign-key integrity;
   - verify semantic identifier uniqueness;
   - test for orphaned records;
   - test consistency between linked storage units, features, and assessments;
   - confirm that spatial layers remain readable in QGIS/GeoPandas;
   - confirm that relational and metadata tables remain easily inspectable in DBeaver.

## Intended Outcome

The output of this notebook should be a **self-describing unified GeoPackage** in which:

- geological and administrative records are linked through explicit relational structure;
- source provenance can be traced from canonical records back to their originating datasets and layers;
- table and field definitions are documented inside the database;
- GeoPackage metadata is valid and internally consistent;
- the resulting structure can later be reproduced programmatically through a dedicated metadata and schema-hardening script.

Notebook 10 therefore serves as the transition from the initial harmonized database created in Notebook 09 to a more formalized and reusable database architecture suitable for downstream modelling, GIS analysis, validation, and publication.

In [11]:
# ---------------------------------------------------------------------------
# Inspect the current unified GeoPackage structure
# ---------------------------------------------------------------------------

from contextlib import closing
from pathlib import Path
import sqlite3
import pandas as pd
import geopandas as gpd
import shutil

# ---------------------------------------------------------------------------
# Input GeoPackage
# ---------------------------------------------------------------------------

GPKG_PATH = Path(
    "../data/processed/unified_storage/canada_geological_storage_unified.gpkg"
).resolve()

if not GPKG_PATH.exists():
    raise FileNotFoundError(f"GeoPackage not found: {GPKG_PATH}")

print(f"GeoPackage: {GPKG_PATH}")

# ---------------------------------------------------------------------------
# Inspect SQLite / GeoPackage tables
# ---------------------------------------------------------------------------

with closing(sqlite3.connect(GPKG_PATH)) as conn:
    database_objects = pd.read_sql_query(
        """
        SELECT
            name,
            type
        FROM sqlite_master
        WHERE type IN ('table', 'view')
        ORDER BY type, name;
        """,
        conn,
    )

    gpkg_contents = pd.read_sql_query(
        """
        SELECT *
        FROM gpkg_contents
        ORDER BY table_name;
        """,
        conn,
    )

    gpkg_geometry_columns = pd.read_sql_query(
        """
        SELECT *
        FROM gpkg_geometry_columns
        ORDER BY table_name;
        """,
        conn,
    )

print("\nDatabase objects:")
display(database_objects)

print("\nGeoPackage contents registry:")
display(gpkg_contents)

print("\nGeoPackage geometry registry:")
display(gpkg_geometry_columns)

# ---------------------------------------------------------------------------
# Canonical tables expected from Notebook 09
# ---------------------------------------------------------------------------

CANONICAL_TABLES = [
    "storage_units",
    "storage_features",
    "storage_assessments",
    "administrative_features",
]

present_tables = set(database_objects["name"])

missing_tables = [
    table
    for table in CANONICAL_TABLES
    if table not in present_tables
]

if missing_tables:
    raise ValueError(
        f"Missing expected canonical table(s): {missing_tables}"
    )

print("\nCanonical tables:")
for table in CANONICAL_TABLES:
    registration = gpkg_contents.loc[
        gpkg_contents["table_name"] == table,
        "data_type",
    ]

    data_type = (
        registration.iloc[0]
        if not registration.empty
        else "NOT REGISTERED"
    )

    print(f"  - {table}: {data_type}")

GeoPackage: C:\Users\aviga\Research\repos\canco2-storage\data\processed\unified_storage\canada_geological_storage_unified.gpkg

Database objects:


,name,type
0,administrative_features,table
1,gpkg_contents,table
2,gpkg_extensions,table
3,gpkg_geometry_columns,table
4,gpkg_ogr_contents,table
5,gpkg_spatial_ref_sys,table
6,gpkg_tile_matrix,table
7,gpkg_tile_matrix_set,table
8,rtree_administrative_features_geom,table
9,rtree_administrative_features_geom_node,table



GeoPackage contents registry:


,table_name,data_type,identifier,description,last_change,min_x,min_y,max_x,max_y,srs_id
0,administrative_features,features,administrative_features,,2026-09-16T03:06:03.304Z,-1.496583e+06,212143.324821,-9.416869e+05,1.059745e+06,3978
1,storage_features,features,storage_features,,2026-09-16T03:06:03.192Z,-2.349109e+06,-706608.096292,3.049300e+06,2.934766e+06,3978



GeoPackage geometry registry:


,table_name,column_name,geometry_type_name,srs_id,z,m
0,administrative_features,geom,MULTIPOLYGON,3978,0,0
1,storage_features,geom,GEOMETRY,3978,0,0



Canonical tables:
  - storage_units: NOT REGISTERED
  - storage_features: features
  - storage_assessments: NOT REGISTERED
  - administrative_features: features


In [6]:
# ---------------------------------------------------------------------------
# Inspect canonical table schemas, indexes, and foreign keys
# ---------------------------------------------------------------------------

CANONICAL_TABLES = [
    "storage_units",
    "storage_features",
    "storage_assessments",
    "administrative_features",
]

schema_inventory = {}
index_inventory = {}
foreign_key_inventory = {}

with closing(sqlite3.connect(GPKG_PATH)) as conn:

    for table in CANONICAL_TABLES:

        # ---------------------------------------------------------------
        # Column definitions
        # ---------------------------------------------------------------

        schema = pd.read_sql_query(
            f'PRAGMA table_info("{table}");',
            conn,
        )

        schema_inventory[table] = schema

        # ---------------------------------------------------------------
        # Index definitions
        # ---------------------------------------------------------------

        indexes = pd.read_sql_query(
            f'PRAGMA index_list("{table}");',
            conn,
        )

        index_inventory[table] = indexes

        # ---------------------------------------------------------------
        # Foreign-key definitions
        # ---------------------------------------------------------------

        foreign_keys = pd.read_sql_query(
            f'PRAGMA foreign_key_list("{table}");',
            conn,
        )

        foreign_key_inventory[table] = foreign_keys

# ---------------------------------------------------------------------------
# Display results
# ---------------------------------------------------------------------------

for table in CANONICAL_TABLES:

    print("\n" + "=" * 80)
    print(table)
    print("=" * 80)

    print("\nColumns:")
    display(schema_inventory[table])

    print("\nIndexes:")
    display(index_inventory[table])

    print("\nForeign keys:")
    display(foreign_key_inventory[table])


storage_units

Columns:


,cid,name,type,notnull,dflt_value,pk
0,0,storage_unit_id,TEXT,0,None,0
1,1,source_dataset,TEXT,0,None,0
2,2,source_unit_id,TEXT,0,None,0
3,3,storage_type,TEXT,0,None,0
4,4,storage_subtype,TEXT,0,None,0
5,5,storage_name,TEXT,0,None,0
6,6,formation,TEXT,0,None,0
7,7,geological_group,TEXT,0,None,0
8,8,basin_name,TEXT,0,None,0
9,9,country,TEXT,0,None,0



Indexes:


,seq,name,unique,origin,partial



Foreign keys:


...

,cid,name,type,notnull,dflt_value,pk
0,0,fid,INTEGER,1,None,1
1,1,geom,MULTIPOLYGON,0,None,0
2,2,administrative_feature_id,TEXT,0,None,0
3,3,parent_administrative_feature_id,TEXT,0,None,0
4,4,source_dataset,TEXT,0,None,0
5,5,source_layer,TEXT,0,None,0
6,6,source_feature_id,TEXT,0,None,0
7,7,administrative_type,TEXT,0,None,0
8,8,agreement_id,TEXT,0,None,0
9,9,tract_id,TEXT,0,None,0



Indexes:


,seq,name,unique,origin,partial



Foreign keys:


,id,seq,table,from,to,on_update,on_delete,match


In [7]:
# ---------------------------------------------------------------------------
# Validate canonical IDs and relational consistency
# ---------------------------------------------------------------------------

ID_COLUMNS = {
    "storage_units": "storage_unit_id",
    "storage_features": "storage_feature_id",
    "storage_assessments": "storage_assessment_id",
    "administrative_features": "administrative_feature_id",
}

id_results = []

with closing(sqlite3.connect(GPKG_PATH)) as conn:

    # -----------------------------------------------------------------------
    # Semantic ID completeness and uniqueness
    # -----------------------------------------------------------------------

    for table, id_column in ID_COLUMNS.items():

        result = pd.read_sql_query(
            f"""
            SELECT
                COUNT(*) AS total_rows,
                SUM(
                    CASE
                        WHEN "{id_column}" IS NULL
                          OR TRIM("{id_column}") = ''
                        THEN 1
                        ELSE 0
                    END
                ) AS missing_ids,
                COUNT(DISTINCT "{id_column}") AS distinct_ids
            FROM "{table}";
            """,
            conn,
        ).iloc[0]

        total_rows = int(result["total_rows"])
        missing_ids = int(result["missing_ids"])
        distinct_ids = int(result["distinct_ids"])

        id_results.append(
            {
                "table": table,
                "id_column": id_column,
                "total_rows": total_rows,
                "missing_ids": missing_ids,
                "distinct_ids": distinct_ids,
                "duplicate_ids": total_rows - missing_ids - distinct_ids,
                "valid_key_candidate": (
                    missing_ids == 0
                    and distinct_ids == total_rows
                ),
            }
        )

    # -----------------------------------------------------------------------
    # Orphan relationship checks
    # -----------------------------------------------------------------------

    orphan_feature_units = pd.read_sql_query(
        """
        SELECT
            f.storage_feature_id,
            f.storage_unit_id
        FROM storage_features AS f
        LEFT JOIN storage_units AS u
            ON f.storage_unit_id = u.storage_unit_id
        WHERE f.storage_unit_id IS NOT NULL
          AND u.storage_unit_id IS NULL;
        """,
        conn,
    )

    orphan_assessment_units = pd.read_sql_query(
        """
        SELECT
            a.storage_assessment_id,
            a.storage_unit_id
        FROM storage_assessments AS a
        LEFT JOIN storage_units AS u
            ON a.storage_unit_id = u.storage_unit_id
        WHERE a.storage_unit_id IS NOT NULL
          AND u.storage_unit_id IS NULL;
        """,
        conn,
    )

    orphan_assessment_features = pd.read_sql_query(
        """
        SELECT
            a.storage_assessment_id,
            a.storage_feature_id
        FROM storage_assessments AS a
        LEFT JOIN storage_features AS f
            ON a.storage_feature_id = f.storage_feature_id
        WHERE a.storage_feature_id IS NOT NULL
          AND f.storage_feature_id IS NULL;
        """,
        conn,
    )

    orphan_admin_parents = pd.read_sql_query(
        """
        SELECT
            a.administrative_feature_id,
            a.parent_administrative_feature_id
        FROM administrative_features AS a
        LEFT JOIN administrative_features AS p
            ON a.parent_administrative_feature_id =
               p.administrative_feature_id
        WHERE a.parent_administrative_feature_id IS NOT NULL
          AND p.administrative_feature_id IS NULL;
        """,
        conn,
    )

    # -----------------------------------------------------------------------
    # Cross-table contradiction check
    # -----------------------------------------------------------------------

    assessment_unit_conflicts = pd.read_sql_query(
        """
        SELECT
            a.storage_assessment_id,
            a.storage_feature_id,
            a.storage_unit_id AS assessment_unit_id,
            f.storage_unit_id AS feature_unit_id
        FROM storage_assessments AS a
        JOIN storage_features AS f
            ON a.storage_feature_id = f.storage_feature_id
        WHERE a.storage_feature_id IS NOT NULL
          AND a.storage_unit_id IS NOT NULL
          AND f.storage_unit_id IS NOT NULL
          AND a.storage_unit_id <> f.storage_unit_id;
        """,
        conn,
    )

id_validation = pd.DataFrame(id_results)

print("Semantic ID validation:")
display(id_validation)

print(f"\nOrphan storage feature → unit links: {len(orphan_feature_units):,}")
display(orphan_feature_units.head(20))

print(f"\nOrphan assessment → unit links: {len(orphan_assessment_units):,}")
display(orphan_assessment_units.head(20))

print(f"\nOrphan assessment → feature links: {len(orphan_assessment_features):,}")
display(orphan_assessment_features.head(20))

print(f"\nOrphan administrative parent links: {len(orphan_admin_parents):,}")
display(orphan_admin_parents.head(20))

print(f"\nAssessment / feature unit conflicts: {len(assessment_unit_conflicts):,}")
display(assessment_unit_conflicts.head(20))

Semantic ID validation:


,table,id_column,total_rows,missing_ids,distinct_ids,duplicate_ids,valid_key_candidate
0,storage_units,storage_unit_id,2843,0,2843,0,True
1,storage_features,storage_feature_id,35320,0,35320,0,True
2,storage_assessments,storage_assessment_id,35245,0,35245,0,True
3,administrative_features,administrative_feature_id,88,0,88,0,True



Orphan storage feature → unit links: 0


,storage_feature_id,storage_unit_id



Orphan assessment → unit links: 0


...

,storage_assessment_id,storage_feature_id



Orphan administrative parent links: 0


,administrative_feature_id,parent_administrative_feature_id



Assessment / feature unit conflicts: 0


,storage_assessment_id,storage_feature_id,assessment_unit_id,feature_unit_id


## Relational integrity validation

The canonical identifiers and existing cross-table relationships were validated before applying database constraints.

### Identifier validation

All four canonical semantic identifiers are:

- non-null;
- non-empty;
- unique within their respective tables.

| Table | Semantic identifier | Rows | Result |
|---|---|---:|---|
| `storage_units` | `storage_unit_id` | 2,843 | Valid unique key |
| `storage_features` | `storage_feature_id` | 35,320 | Valid unique key |
| `storage_assessments` | `storage_assessment_id` | 35,245 | Valid unique key |
| `administrative_features` | `administrative_feature_id` | 88 | Valid unique key |

The spatial tables also contain GeoPackage-generated `fid` primary keys. These will remain the technical row identifiers required by the GeoPackage structure, while the canonical IDs will serve as stable domain identifiers.

### Relationship validation

No orphaned or contradictory relationships were identified:

- `storage_features.storage_unit_id` → `storage_units.storage_unit_id`: **0 orphaned records**
- `storage_assessments.storage_unit_id` → `storage_units.storage_unit_id`: **0 orphaned records**
- `storage_assessments.storage_feature_id` → `storage_features.storage_feature_id`: **0 orphaned records**
- `administrative_features.parent_administrative_feature_id` → `administrative_features.administrative_feature_id`: **0 orphaned records**
- assessment-to-feature unit inconsistencies: **0 records**

This confirms that the relationships established during harmonization already behave as a consistent relational model, even though they are not yet formally enforced by SQLite.

## Intended relational structure

The canonical domain relationships will therefore be formalized as:

```text
storage_units
    storage_unit_id
        │
        ├──────────< storage_features.storage_unit_id
        │
        └──────────< storage_assessments.storage_unit_id

storage_features
    storage_feature_id
        │
        └──────────< storage_assessments.storage_feature_id

administrative_features
    administrative_feature_id
        │
        └──────────< parent_administrative_feature_id
                      (self-referencing hierarchy)

In [8]:
# ---------------------------------------------------------------------------
# Inspect nullability and relationship cardinalities
# ---------------------------------------------------------------------------

with closing(sqlite3.connect(GPKG_PATH)) as conn:

    # -----------------------------------------------------------------------
    # Null patterns in relational fields
    # -----------------------------------------------------------------------

    nullability_summary = pd.read_sql_query(
        """
        SELECT
            'storage_features' AS table_name,
            COUNT(*) AS total_rows,
            SUM(CASE WHEN storage_unit_id IS NULL THEN 1 ELSE 0 END) AS null_unit_id,
            NULL AS null_feature_id,
            NULL AS null_parent_id
        FROM storage_features

        UNION ALL

        SELECT
            'storage_assessments' AS table_name,
            COUNT(*) AS total_rows,
            SUM(CASE WHEN storage_unit_id IS NULL THEN 1 ELSE 0 END) AS null_unit_id,
            SUM(CASE WHEN storage_feature_id IS NULL THEN 1 ELSE 0 END) AS null_feature_id,
            NULL AS null_parent_id
        FROM storage_assessments

        UNION ALL

        SELECT
            'administrative_features' AS table_name,
            COUNT(*) AS total_rows,
            NULL AS null_unit_id,
            NULL AS null_feature_id,
            SUM(
                CASE
                    WHEN parent_administrative_feature_id IS NULL
                    THEN 1
                    ELSE 0
                END
            ) AS null_parent_id
        FROM administrative_features;
        """,
        conn,
    )

    # -----------------------------------------------------------------------
    # Assessment scope and linkage patterns
    # -----------------------------------------------------------------------

    assessment_linkage = pd.read_sql_query(
        """
        SELECT
            assessment_scope,
            COUNT(*) AS assessments,
            SUM(CASE WHEN storage_unit_id IS NOT NULL THEN 1 ELSE 0 END)
                AS with_unit_id,
            SUM(CASE WHEN storage_feature_id IS NOT NULL THEN 1 ELSE 0 END)
                AS with_feature_id,
            SUM(
                CASE
                    WHEN storage_unit_id IS NOT NULL
                     AND storage_feature_id IS NOT NULL
                    THEN 1
                    ELSE 0
                END
            ) AS with_both,
            SUM(
                CASE
                    WHEN storage_unit_id IS NOT NULL
                     AND storage_feature_id IS NULL
                    THEN 1
                    ELSE 0
                END
            ) AS unit_only,
            SUM(
                CASE
                    WHEN storage_unit_id IS NULL
                     AND storage_feature_id IS NOT NULL
                    THEN 1
                    ELSE 0
                END
            ) AS feature_only,
            SUM(
                CASE
                    WHEN storage_unit_id IS NULL
                     AND storage_feature_id IS NULL
                    THEN 1
                    ELSE 0
                END
            ) AS unlinked
        FROM storage_assessments
        GROUP BY assessment_scope
        ORDER BY assessments DESC;
        """,
        conn,
    )

    # -----------------------------------------------------------------------
    # Parent-child structure in administrative features
    # -----------------------------------------------------------------------

    administrative_hierarchy = pd.read_sql_query(
        """
        SELECT
            administrative_type,
            COUNT(*) AS features,
            SUM(
                CASE
                    WHEN parent_administrative_feature_id IS NOT NULL
                    THEN 1
                    ELSE 0
                END
            ) AS child_features,
            SUM(
                CASE
                    WHEN parent_administrative_feature_id IS NULL
                    THEN 1
                    ELSE 0
                END
            ) AS root_features
        FROM administrative_features
        GROUP BY administrative_type
        ORDER BY features DESC;
        """,
        conn,
    )

    # -----------------------------------------------------------------------
    # Storage unit cardinalities
    # -----------------------------------------------------------------------

    unit_cardinality = pd.read_sql_query(
        """
        SELECT
            u.storage_unit_id,
            COUNT(DISTINCT f.storage_feature_id) AS feature_count,
            COUNT(DISTINCT a.storage_assessment_id) AS assessment_count
        FROM storage_units AS u
        LEFT JOIN storage_features AS f
            ON u.storage_unit_id = f.storage_unit_id
        LEFT JOIN storage_assessments AS a
            ON u.storage_unit_id = a.storage_unit_id
        GROUP BY u.storage_unit_id;
        """,
        conn,
    )

print("Nullability summary:")
display(nullability_summary)

print("\nAssessment linkage patterns:")
display(assessment_linkage)

print("\nAdministrative hierarchy:")
display(administrative_hierarchy)

print("\nStorage-unit relationship cardinality summary:")
display(
    unit_cardinality[
        ["feature_count", "assessment_count"]
    ].describe()
)

Nullability summary:


,table_name,total_rows,null_unit_id,null_feature_id,null_parent_id
0,storage_features,35320,0.0,NaN,NaN
1,storage_assessments,35245,0.0,1263.0,NaN
2,administrative_features,88,NaN,NaN,43.0



Assessment linkage patterns:


,assessment_scope,assessments,with_unit_id,with_feature_id,with_both,unit_only,feature_only,unlinked
0,feature,33982,33982,33982,33982,0,0,0
1,unit,1263,1263,0,0,1263,0,0



Administrative hierarchy:


,administrative_type,features,child_features,root_features
0,carbon_sequestration_agreement_tract,45,45,0
1,carbon_sequestration_agreement,43,0,43



Storage-unit relationship cardinality summary:


,feature_count,assessment_count
count,2843.000000,2843.000000
mean,12.423496,12.397116
std,191.291258,191.292760
min,1.000000,1.000000
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,1.000000,1.000000
max,8218.000000,8218.000000


In [9]:
# ---------------------------------------------------------------------------
# Validate conditional relationship rules and inspect extreme cardinalities
# ---------------------------------------------------------------------------

with closing(sqlite3.connect(GPKG_PATH)) as conn:

    # -----------------------------------------------------------------------
    # Assessment-scope consistency
    # -----------------------------------------------------------------------

    invalid_assessment_scope_links = pd.read_sql_query(
        """
        SELECT
            storage_assessment_id,
            assessment_scope,
            storage_unit_id,
            storage_feature_id
        FROM storage_assessments
        WHERE
            (
                assessment_scope = 'feature'
                AND storage_feature_id IS NULL
            )
            OR
            (
                assessment_scope = 'unit'
                AND storage_feature_id IS NOT NULL
            )
            OR
            assessment_scope NOT IN ('feature', 'unit')
            OR
            assessment_scope IS NULL;
        """,
        conn,
    )

    # -----------------------------------------------------------------------
    # Administrative hierarchy consistency
    # -----------------------------------------------------------------------

    invalid_admin_hierarchy = pd.read_sql_query(
        """
        SELECT
            administrative_feature_id,
            parent_administrative_feature_id,
            administrative_type
        FROM administrative_features
        WHERE
            (
                administrative_type = 'carbon_sequestration_agreement'
                AND parent_administrative_feature_id IS NOT NULL
            )
            OR
            (
                administrative_type = 'carbon_sequestration_agreement_tract'
                AND parent_administrative_feature_id IS NULL
            );
        """,
        conn,
    )

    # -----------------------------------------------------------------------
    # Highest-cardinality storage units
    # -----------------------------------------------------------------------

    high_cardinality_units = pd.read_sql_query(
        """
        SELECT
            u.storage_unit_id,
            u.source_dataset,
            u.source_unit_id,
            u.storage_type,
            u.storage_subtype,
            u.storage_name,
            COUNT(DISTINCT f.storage_feature_id) AS feature_count,
            COUNT(DISTINCT a.storage_assessment_id) AS assessment_count
        FROM storage_units AS u
        LEFT JOIN storage_features AS f
            ON u.storage_unit_id = f.storage_unit_id
        LEFT JOIN storage_assessments AS a
            ON u.storage_unit_id = a.storage_unit_id
        GROUP BY
            u.storage_unit_id,
            u.source_dataset,
            u.source_unit_id,
            u.storage_type,
            u.storage_subtype,
            u.storage_name
        ORDER BY feature_count DESC
        LIMIT 20;
        """,
        conn,
    )

print(
    "Invalid assessment-scope relationships:",
    len(invalid_assessment_scope_links),
)
display(invalid_assessment_scope_links)

print(
    "\nInvalid administrative hierarchy relationships:",
    len(invalid_admin_hierarchy),
)
display(invalid_admin_hierarchy)

print("\nHighest-cardinality storage units:")
display(high_cardinality_units)

Invalid assessment-scope relationships: 0


,storage_assessment_id,assessment_scope,storage_unit_id,storage_feature_id



Invalid administrative hierarchy relationships: 0


,administrative_feature_id,parent_administrative_feature_id,administrative_type



Highest-cardinality storage units:


,storage_unit_id,source_dataset,source_unit_id,storage_type,storage_subtype,storage_name,feature_count,assessment_count
0,NAT_SAL_61c04b2208a1,NATCARB,NATCARB|PCOR|SALINE|BASAL CAMBRIAN,saline_aquifer,None,Basal Cambrian,8218,8218
1,NAT_SAL_12a3f046cd96,NATCARB,NATCARB|PCOR|SALINE|ELK POINT GROUP,saline_aquifer,None,Elk Point Group,3520,3520
2,NAT_SAL_d89a2c04173c,NATCARB,NATCARB|PCOR|SALINE|BEAVERHILL LAKE GROUP,saline_aquifer,None,Beaverhill Lake Group,2278,2278
3,NAT_SAL_e6ded5c9bdb2,NATCARB,NATCARB|PCOR|SALINE|WINTERBURN GROUP,saline_aquifer,None,Winterburn Group,2158,2158
4,NAT_SAL_c0379c47d5ed,NATCARB,NATCARB|PCOR|SALINE|RUNDLE GROUP,saline_aquifer,None,Rundle Group,1685,1685
5,NAT_SAL_eee374fd1882,NATCARB,NATCARB|PCOR|SALINE|WOODBEND GROUP,saline_aquifer,None,Woodbend Group,1592,1592
6,NAT_SAL_a33021630285,NATCARB,NATCARB|PCOR|SALINE|VIKING,saline_aquifer,None,Viking,1350,1350
7,NAT_SAL_9f1a60ae2435,NATCARB,NATCARB|WESTCARB|SALINE|ROCKY MOUNTAIN FOOTHILLS,saline_aquifer,None,Rocky Mountain Foothills,1328,1328
8,ATL_gsc8996_pictou,ATLANTIC_COS,gsc8996_pictou,NaN,None,Pictou Group,1166,1166
9,NAT_SAL_4389faa20243,NATCARB,NATCARB|WESTCARB|SALINE|NECHAKO BASIN,saline_aquifer,None,Nechako Basin,889,889


## Relational cardinality and nullability decisions

Additional validation was performed to determine which relationships represent mandatory schema rules and which should remain nullable.

### Storage features

All `storage_features` records contain a valid `storage_unit_id`.

Therefore:

- every spatial storage feature must belong to a canonical storage unit;
- `storage_features.storage_unit_id` can be treated as **NOT NULL**;
- the relationship is many-to-one:

```text
storage_units 1 ─────< N storage_features

In [12]:
# ---------------------------------------------------------------------------
# Create a timestamped backup before modifying the GeoPackage
# ---------------------------------------------------------------------------

BACKUP_DIR = GPKG_PATH.parent / "backups"
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

BACKUP_PATH = (
    BACKUP_DIR
    / f"{GPKG_PATH.stem}_pre_schema_hardening_{timestamp}{GPKG_PATH.suffix}"
)

shutil.copy2(GPKG_PATH, BACKUP_PATH)

if not BACKUP_PATH.exists():
    raise FileNotFoundError(
        f"Backup was not created successfully: {BACKUP_PATH}"
    )

original_size = GPKG_PATH.stat().st_size
backup_size = BACKUP_PATH.stat().st_size

if original_size != backup_size:
    raise ValueError(
        "Backup size does not match the original GeoPackage."
    )

print("Backup created successfully.")
print(f"Original: {GPKG_PATH}")
print(f"Backup:   {BACKUP_PATH}")
print(f"Size:     {original_size / 1e6:.2f} MB")

Backup created successfully.
Original: C:\Users\aviga\Research\repos\canco2-storage\data\processed\unified_storage\canada_geological_storage_unified.gpkg
Backup:   C:\Users\aviga\Research\repos\canco2-storage\data\processed\unified_storage\backups\canada_geological_storage_unified_pre_schema_hardening_20260916_115446.gpkg
Size:     56.94 MB


In [13]:
# ---------------------------------------------------------------------------
# Register canonical attribute tables and add table descriptions
# ---------------------------------------------------------------------------

TABLE_DESCRIPTIONS = {
    "storage_units": (
        "Canonical conceptual geological CO2 storage units harmonized from "
        "source geological storage datasets."
    ),
    "storage_features": (
        "Spatial representations of canonical geological CO2 storage units, "
        "preserving source-layer feature provenance."
    ),
    "storage_assessments": (
        "Feature-level and unit-level geological CO2 storage assessments, "
        "including capacity and reservoir properties where available."
    ),
    "administrative_features": (
        "Spatial administrative and tenure features relevant to geological "
        "CO2 storage, including hierarchical agreement and tract records."
    ),
}

# ---------------------------------------------------------------------------
# Write changes
# ---------------------------------------------------------------------------

with closing(sqlite3.connect(GPKG_PATH)) as conn:
    conn.execute("PRAGMA foreign_keys = ON;")

    try:
        conn.execute("BEGIN;")

        # Register non-spatial canonical tables as GeoPackage attribute tables
        for table_name in ["storage_units", "storage_assessments"]:

            existing = conn.execute(
                """
                SELECT COUNT(*)
                FROM gpkg_contents
                WHERE table_name = ?;
                """,
                (table_name,),
            ).fetchone()[0]

            if existing == 0:
                conn.execute(
                    """
                    INSERT INTO gpkg_contents (
                        table_name,
                        data_type,
                        identifier,
                        description,
                        last_change,
                        min_x,
                        min_y,
                        max_x,
                        max_y,
                        srs_id
                    )
                    VALUES (
                        ?,
                        'attributes',
                        ?,
                        ?,
                        strftime('%Y-%m-%dT%H:%M:%fZ', 'now'),
                        NULL,
                        NULL,
                        NULL,
                        NULL,
                        NULL
                    );
                    """,
                    (
                        table_name,
                        table_name,
                        TABLE_DESCRIPTIONS[table_name],
                    ),
                )

        # Add/update descriptions for all canonical tables
        for table_name, description in TABLE_DESCRIPTIONS.items():
            conn.execute(
                """
                UPDATE gpkg_contents
                SET description = ?
                WHERE table_name = ?;
                """,
                (description, table_name),
            )

        conn.commit()

    except Exception:
        conn.rollback()
        raise

# ---------------------------------------------------------------------------
# Reopen and verify
# ---------------------------------------------------------------------------

with closing(sqlite3.connect(GPKG_PATH)) as conn:
    canonical_contents = pd.read_sql_query(
        """
        SELECT
            table_name,
            data_type,
            identifier,
            description,
            srs_id
        FROM gpkg_contents
        WHERE table_name IN (
            'storage_units',
            'storage_features',
            'storage_assessments',
            'administrative_features'
        )
        ORDER BY table_name;
        """,
        conn,
    )

display(canonical_contents)

,table_name,data_type,identifier,description,srs_id
0,administrative_features,features,administrative_features,Spatial administrative and tenure features rel...,3978.0
1,storage_assessments,attributes,storage_assessments,Feature-level and unit-level geological CO2 st...,NaN
2,storage_features,features,storage_features,Spatial representations of canonical geologica...,3978.0
3,storage_units,attributes,storage_units,Canonical conceptual geological CO2 storage un...,NaN


In [14]:
# ---------------------------------------------------------------------------
# Create semantic indexes and uniqueness constraints
# ---------------------------------------------------------------------------

with closing(sqlite3.connect(GPKG_PATH)) as conn:
    conn.execute("PRAGMA foreign_keys = ON;")

    try:
        conn.execute("BEGIN;")

        # Canonical unique identifiers
        conn.execute(
            """
            CREATE UNIQUE INDEX IF NOT EXISTS
            uq_storage_features_storage_feature_id
            ON storage_features(storage_feature_id);
            """
        )

        conn.execute(
            """
            CREATE UNIQUE INDEX IF NOT EXISTS
            uq_administrative_features_administrative_feature_id
            ON administrative_features(administrative_feature_id);
            """
        )

        # Relationship indexes
        conn.execute(
            """
            CREATE INDEX IF NOT EXISTS
            ix_storage_features_storage_unit_id
            ON storage_features(storage_unit_id);
            """
        )

        conn.execute(
            """
            CREATE INDEX IF NOT EXISTS
            ix_storage_assessments_storage_unit_id
            ON storage_assessments(storage_unit_id);
            """
        )

        conn.execute(
            """
            CREATE INDEX IF NOT EXISTS
            ix_storage_assessments_storage_feature_id
            ON storage_assessments(storage_feature_id);
            """
        )

        conn.execute(
            """
            CREATE INDEX IF NOT EXISTS
            ix_administrative_features_parent_id
            ON administrative_features(parent_administrative_feature_id);
            """
        )

        conn.commit()

    except Exception:
        conn.rollback()
        raise

# ---------------------------------------------------------------------------
# Verify indexes
# ---------------------------------------------------------------------------

index_rows = []

with closing(sqlite3.connect(GPKG_PATH)) as conn:
    for table in CANONICAL_TABLES:
        indexes = pd.read_sql_query(
            f'PRAGMA index_list("{table}");',
            conn,
        )

        if not indexes.empty:
            indexes.insert(0, "table_name", table)
            index_rows.append(indexes)

index_summary = (
    pd.concat(index_rows, ignore_index=True)
    if index_rows
    else pd.DataFrame()
)

display(index_summary)

,table_name,seq,name,unique,origin,partial
0,storage_features,0,ix_storage_features_storage_unit_id,0,c,0
1,storage_features,1,uq_storage_features_storage_feature_id,1,c,0
2,storage_assessments,0,ix_storage_assessments_storage_feature_id,0,c,0
3,storage_assessments,1,ix_storage_assessments_storage_unit_id,0,c,0
4,administrative_features,0,ix_administrative_features_parent_id,0,c,0
5,administrative_features,1,uq_administrative_features_administrative_feat...,1,c,0


In [15]:
# ---------------------------------------------------------------------------
# Harden non-spatial canonical tables
# ---------------------------------------------------------------------------

with closing(sqlite3.connect(GPKG_PATH)) as conn:

    # Foreign-key enforcement is disabled only during the controlled table rebuild.
    conn.execute("PRAGMA foreign_keys = OFF;")

    try:
        conn.execute("BEGIN;")

        # -------------------------------------------------------------------
        # Rebuild storage_units
        # -------------------------------------------------------------------

        conn.execute(
            """
            CREATE TABLE storage_units_new (
                storage_unit_id TEXT NOT NULL PRIMARY KEY,
                source_dataset TEXT,
                source_unit_id TEXT,
                storage_type TEXT,
                storage_subtype TEXT,
                storage_name TEXT,
                formation TEXT,
                geological_group TEXT,
                basin_name TEXT,
                country TEXT,
                province_territory TEXT,
                land_status TEXT,
                assessment_type TEXT,
                data_class TEXT,
                capacity_data INTEGER,
                capacity_status TEXT,
                injectivity_status TEXT,
                co2_phase TEXT
            );
            """
        )

        conn.execute(
            """
            INSERT INTO storage_units_new
            SELECT *
            FROM storage_units;
            """
        )

        # -------------------------------------------------------------------
        # Rebuild storage_assessments
        # -------------------------------------------------------------------

        conn.execute(
            """
            CREATE TABLE storage_assessments_new (
                storage_assessment_id TEXT NOT NULL PRIMARY KEY,

                storage_feature_id TEXT,
                storage_unit_id TEXT NOT NULL,

                source_dataset TEXT,
                source_layer TEXT,

                assessment_scope TEXT NOT NULL,
                assessment_type TEXT,

                storage_p10_tonnes REAL,
                storage_p50_tonnes REAL,
                storage_p90_tonnes REAL,
                theoretical_storage_tonnes REAL,
                effective_storage_tonnes REAL,

                capacity_basis TEXT,
                capacity_method TEXT,
                capacity_status TEXT,

                depth_m REAL,
                thickness_m REAL,
                pressure_mpa REAL,
                temperature_c REAL,
                porosity_fraction REAL,
                permeability_md REAL,
                salinity_tds_ppm REAL,

                reservoir_cos REAL,
                seal_cos REAL,
                trap_cos REAL,
                total_cos REAL,

                co2_phase TEXT,

                FOREIGN KEY (storage_unit_id)
                    REFERENCES storage_units(storage_unit_id),

                FOREIGN KEY (storage_feature_id)
                    REFERENCES storage_features(storage_feature_id),

                CHECK (
                    (
                        assessment_scope = 'feature'
                        AND storage_feature_id IS NOT NULL
                    )
                    OR
                    (
                        assessment_scope = 'unit'
                        AND storage_feature_id IS NULL
                    )
                )
            );
            """
        )

        conn.execute(
            """
            INSERT INTO storage_assessments_new
            SELECT *
            FROM storage_assessments;
            """
        )

        # -------------------------------------------------------------------
        # Confirm row preservation before replacing original tables
        # -------------------------------------------------------------------

        old_units = conn.execute(
            "SELECT COUNT(*) FROM storage_units;"
        ).fetchone()[0]

        new_units = conn.execute(
            "SELECT COUNT(*) FROM storage_units_new;"
        ).fetchone()[0]

        old_assessments = conn.execute(
            "SELECT COUNT(*) FROM storage_assessments;"
        ).fetchone()[0]

        new_assessments = conn.execute(
            "SELECT COUNT(*) FROM storage_assessments_new;"
        ).fetchone()[0]

        if old_units != new_units:
            raise ValueError(
                f"storage_units row mismatch: {old_units} != {new_units}"
            )

        if old_assessments != new_assessments:
            raise ValueError(
                "storage_assessments row mismatch: "
                f"{old_assessments} != {new_assessments}"
            )

        # -------------------------------------------------------------------
        # Replace original tables
        # -------------------------------------------------------------------

        conn.execute("DROP TABLE storage_assessments;")
        conn.execute("DROP TABLE storage_units;")

        conn.execute(
            "ALTER TABLE storage_units_new RENAME TO storage_units;"
        )

        conn.execute(
            """
            ALTER TABLE storage_assessments_new
            RENAME TO storage_assessments;
            """
        )

        # -------------------------------------------------------------------
        # Recreate relationship indexes removed with the old assessment table
        # -------------------------------------------------------------------

        conn.execute(
            """
            CREATE INDEX ix_storage_assessments_storage_unit_id
            ON storage_assessments(storage_unit_id);
            """
        )

        conn.execute(
            """
            CREATE INDEX ix_storage_assessments_storage_feature_id
            ON storage_assessments(storage_feature_id);
            """
        )

        conn.commit()

    except Exception:
        conn.rollback()
        raise

# ---------------------------------------------------------------------------
# Reopen with FK enforcement and validate hardened schema
# ---------------------------------------------------------------------------

with closing(sqlite3.connect(GPKG_PATH)) as conn:
    conn.execute("PRAGMA foreign_keys = ON;")

    units_schema = pd.read_sql_query(
        'PRAGMA table_info("storage_units");',
        conn,
    )

    assessments_schema = pd.read_sql_query(
        'PRAGMA table_info("storage_assessments");',
        conn,
    )

    assessment_foreign_keys = pd.read_sql_query(
        'PRAGMA foreign_key_list("storage_assessments");',
        conn,
    )

    foreign_key_check = pd.read_sql_query(
        "PRAGMA foreign_key_check;",
        conn,
    )

print("storage_units schema:")
display(units_schema)

print("\nstorage_assessments schema:")
display(assessments_schema)

print("\nstorage_assessments foreign keys:")
display(assessment_foreign_keys)

print("\nForeign-key violations:")
display(foreign_key_check)

storage_units schema:


,cid,name,type,notnull,dflt_value,pk
0,0,storage_unit_id,TEXT,1,None,1
1,1,source_dataset,TEXT,0,None,0
2,2,source_unit_id,TEXT,0,None,0
3,3,storage_type,TEXT,0,None,0
4,4,storage_subtype,TEXT,0,None,0
5,5,storage_name,TEXT,0,None,0
6,6,formation,TEXT,0,None,0
7,7,geological_group,TEXT,0,None,0
8,8,basin_name,TEXT,0,None,0
9,9,country,TEXT,0,None,0



storage_assessments schema:


,cid,name,type,notnull,dflt_value,pk
0,0,storage_assessment_id,TEXT,1,None,1
1,1,storage_feature_id,TEXT,0,None,0
2,2,storage_unit_id,TEXT,1,None,0
3,3,source_dataset,TEXT,0,None,0
4,4,source_layer,TEXT,0,None,0
5,5,assessment_scope,TEXT,1,None,0
6,6,assessment_type,TEXT,0,None,0
7,7,storage_p10_tonnes,REAL,0,None,0
8,8,storage_p50_tonnes,REAL,0,None,0
9,9,storage_p90_tonnes,REAL,0,None,0



storage_assessments foreign keys:


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,storage_features,storage_feature_id,storage_feature_id,NO ACTION,NO ACTION,NONE
1,1,0,storage_units,storage_unit_id,storage_unit_id,NO ACTION,NO ACTION,NONE



Foreign-key violations:


,table,rowid,parent,fkid


In [16]:
# ---------------------------------------------------------------------------
# Inspect spatial-table DDL and GeoPackage registrations
# ---------------------------------------------------------------------------

with closing(sqlite3.connect(GPKG_PATH)) as conn:

    spatial_table_ddl = pd.read_sql_query(
        """
        SELECT
            name,
            sql
        FROM sqlite_master
        WHERE type = 'table'
          AND name IN (
              'storage_features',
              'administrative_features'
          )
        ORDER BY name;
        """,
        conn,
    )

    geometry_registration = pd.read_sql_query(
        """
        SELECT *
        FROM gpkg_geometry_columns
        WHERE table_name IN (
            'storage_features',
            'administrative_features'
        )
        ORDER BY table_name;
        """,
        conn,
    )

    spatial_extensions = pd.read_sql_query(
        """
        SELECT *
        FROM gpkg_extensions
        WHERE table_name IN (
            'storage_features',
            'administrative_features'
        )
        ORDER BY table_name, extension_name;
        """,
        conn,
    )

print("Spatial table DDL:")
display(spatial_table_ddl)

print("\nGeometry registration:")
display(geometry_registration)

print("\nSpatial extensions:")
display(spatial_extensions)

Spatial table DDL:


,name,sql
0,administrative_features,"CREATE TABLE ""administrative_features"" ( ""fid""..."
1,storage_features,"CREATE TABLE ""storage_features"" ( ""fid"" INTEGE..."



Geometry registration:


,table_name,column_name,geometry_type_name,srs_id,z,m
0,administrative_features,geom,MULTIPOLYGON,3978,0,0
1,storage_features,geom,GEOMETRY,3978,0,0



Spatial extensions:


,table_name,column_name,extension_name,definition,scope
0,administrative_features,geom,gpkg_rtree_index,http://www.geopackage.org/spec120/#extension_r...,write-only
1,storage_features,geom,gpkg_rtree_index,http://www.geopackage.org/spec120/#extension_r...,write-only


## Spatial table hardening strategy

Inspection of the two canonical spatial tables confirms that they are standard GeoPackage feature tables registered through:

- `gpkg_contents`;
- `gpkg_geometry_columns`;
- `gpkg_extensions`;
- GeoPackage RTree spatial-index tables.

Both spatial layers use GeoPackage-managed `fid` primary keys and have active `gpkg_rtree_index` extensions.

### Decision

The spatial feature tables will **not be rebuilt** during Notebook 10.

Rebuilding these tables solely to introduce domain-level foreign-key declarations would require recreating and validating:

- GeoPackage feature registration;
- geometry-column registration;
- spatial reference metadata;
- RTree spatial indexes;
- GeoPackage spatial-index triggers;
- layer extents and related metadata.

The canonical semantic identifiers already satisfy uniqueness requirements and are now protected with unique indexes:

- `storage_features.storage_feature_id`
- `administrative_features.administrative_feature_id`

The GeoPackage-generated `fid` fields will therefore remain the physical primary keys of the spatial tables, while the semantic identifiers remain stable domain identifiers.

### Remaining spatial relationships

Two logical relationships still require enforcement:

```text
storage_features.storage_unit_id
    → storage_units.storage_unit_id

# 10 — Relational Structure in the Unified GeoPackage

## Purpose

Notebook 09 established the first unified GeoPackage structure for the Canadian geological CO₂ storage database.

During review of that structure, this notebook explored whether the unified GeoPackage should also be hardened into a more explicitly relational SQLite database using primary keys, foreign keys, unique constraints, indexes, and validation triggers.

The main conclusion is that the existing database already contains a coherent **logical relational structure**, but that a GeoPackage does not necessarily need to encode the full domain ontology through SQL constraints in order to function effectively as a portable GIS database.

This notebook is therefore retained primarily as a **schema validation and design exercise** rather than as a permanent schema-migration workflow.

## Existing logical relationships

The unified database contains four canonical tables:

- `storage_units`
- `storage_features`
- `storage_assessments`
- `administrative_features`

Their intended logical relationships are:

```text
storage_units
    storage_unit_id
        │
        ├──────────< storage_features.storage_unit_id
        │
        └──────────< storage_assessments.storage_unit_id

storage_features
    storage_feature_id
        │
        └──────────< storage_assessments.storage_feature_id

administrative_features
    administrative_feature_id
        │
        └──────────< parent_administrative_feature_id
                      (self-referencing hierarchy)